In [1]:
%matplotlib qt
import numpy as np
import matplotlib.pyplot as plt
from astropy.io import fits
from datetime import datetime
import glob
from reprojection import *
from show import *
from utils import *
from scipy.ndimage import gaussian_filter

In [9]:
files = np.array(sorted(glob.glob('/home/ulyanov/data/solo/phi/2026/blos_/*')))
dates = np.array([datetime.fromisoformat(file.split('/')[-1].split('_')[-3]) for file in files])


#t = np.where(np.all([dates > datetime(2026,7,22),
#                     dates < datetime(2026,8,31)], axis=0))[0] # CR2314

#t = np.where(np.all([dates > datetime(2026,8,31),
#                     dates < datetime(2026,9,29)], axis=0))[0] # CR2315

#t = np.where(np.all([dates > datetime(2026,7,8),
#                     dates < datetime(2026,8,8)], axis=0))[0]

t = np.where(np.all([dates > datetime(2026,9,14),
                     dates < datetime(2026,10,13)], axis=0))[0]

files = files[t]
files[0], files[-1], len(files)

(np.str_('/home/ulyanov/data/solo/phi/2026/blos_/phi-fdt-blos_20260914T123003_V202610202132C_0649140501.fits'),
 np.str_('/home/ulyanov/data/solo/phi/2026/blos_/phi-fdt-blos_20261012T093003_V202610211218C_0650120503.fits'),
 121)

In [17]:
data, alpha = make_map(files, correct_dr=True, sine_lat=True, minmu=0., delta_t=10, correct_bias=True)

/home/ulyanov/PycharmProjects/reprojection/src/reprojection.py:153: RuntimeWarning: invalid value encountered in divide
  mean_mu += np.nan_to_num((mu - mean_mu) * weight / coverage)
/home/ulyanov/PycharmProjects/reprojection/src/reprojection.py:167: RuntimeWarning: invalid value encountered in divide
  Q = cov_data_mu / var_mu


In [18]:
#import seaborn as sns
#mycmap = sns.color_palette("vlag", as_cmap=True)
show_map(data, vmin=-100, vmax=100, sine_lat=True, cmap='seismic')

In [19]:
nx = len(data)
#lat = (np.arange(nx) / (nx - 1) * 2 - 1) * 90
lat = np.arcsin(np.arange(nx) / (nx - 1) * 2 - 1) * 180 / np.pi
#flux = np.nanmean(data, axis=1)
flux = np.nanmean(data * alpha, axis=1) / np.nanmean(alpha, axis=1)

plt.figure(figsize=(10,8))
plt.plot(lat, flux)

plt.xlim(-90,90)
plt.ylim(-10,10)
plt.grid(True)
plt.tight_layout()

/tmp/ipykernel_33235/1563973450.py:5: RuntimeWarning: Mean of empty slice
  flux = np.nanmean(data * alpha, axis=1) / np.nanmean(alpha, axis=1)


In [20]:
file = '/home/ulyanov/data/sdo/hmi/synoptic_maps/hmi.Synoptic_Mr_720s.2315.synopMr.fits'

with fits.open(file) as hdul:
    header = hdul[0].header.copy()
    data_ = hdul[0].data.copy()

data_ = rebin(data_, 2)
#data_ = gaussian_filter(data_, 2, truncate=3)
header['T_OBS']

'2026.09.12_09:28:02_TAI'

In [21]:
show_map(data_, vmin=-100, vmax=100, sine_lat=True, cmap='seismic')

In [22]:
flux_ = np.nanmean(data_ * alpha, axis=1) / np.nanmean(alpha, axis=1)
#flux_ = np.nanmean(data_, axis=1)

nx, ny = data_.shape
lat_ = np.arcsin(np.arange(nx) / (nx - 1) * 2 - 1) * 180 / np.pi

plt.figure(figsize=(10,8))
plt.plot(lat_, flux_, label='HMI')
plt.plot(lat, flux, label='FDT')

plt.xlabel('Latitude, degrees')
plt.ylabel(r'Flux, Mx/cm$^2$')
plt.xlim(-90,90)

plt.xlim(-90,90)
plt.ylim(-10,10)
plt.grid(True)
plt.legend()
plt.tight_layout()

/tmp/ipykernel_33235/15556052.py:1: RuntimeWarning: Mean of empty slice
  flux_ = np.nanmean(data_ * alpha, axis=1) / np.nanmean(alpha, axis=1)
